# Prueba breve de preentrenamiento del modelo de 50M

Entrena durante unas pocas actualizaciones sobre una porción pequeña de FineWeb-Edu, evalúa, genera texto y guarda un checkpoint local. Funciona con CUDA, Apple MPS o CPU.

In [1]:
## Save checpoints of validation for plotting

In [2]:
from pathlib import Path
import subprocess
import sys

PROJECT_ROOT = Path("/workspace/notebooks")

assert (PROJECT_ROOT / "pyproject.toml").is_file()
assert (PROJECT_ROOT / "src" / "llm_mini_lab").is_dir(), (
    "No existe /workspace/notebooks/src/llm_mini_lab"
)

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-e",
    f"{PROJECT_ROOT}[training]",
    "jupyterlab",
    "ipywidgets",
])

print("Instalado correctamente desde:", PROJECT_ROOT)

Obtaining file:///workspace/notebooks
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 54.1 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 67.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 947.5/947.5 kB 12.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 11.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 29.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 57.4 MB/s  0:00

Instalado correctamente desde: /workspace/notebooks


In [1]:
%pip install -q -e ".[training]" jupyterlab ipywidgets

ERROR: file:///workspace/notebooks/notebooks does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import sys

import torch
import torch.nn.functional as F
import tiktoken

PROJECT_ROOT = next((path for path in (Path.cwd(), *Path.cwd().parents)
                     if (path / 'src' / 'llm_mini_lab').is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('No se encontró la raíz del proyecto llm-mini-lab')
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from llm_mini_lab.models import GPTModel
from llm_mini_lab.training import (
    GPT_CONFIG_50M, create_dataloader_smollm, generate_text_simple,
    text_to_token_ids, token_ids_to_text,
)

if torch.cuda.is_available():
    device = torch.device('cuda')
elif getattr(torch.backends, 'mps', None) and torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

print('PyTorch:', torch.__version__)
print('Dispositivo:', device)
if device.type == 'cuda':
    torch.backends.cuda.matmul.allow_tf32 = True
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM (GiB):', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))

PyTorch: 2.11.0+cu128
Dispositivo: cuda
GPU: NVIDIA GeForce RTX 3060
VRAM (GiB): 11.6


In [3]:
SEED = 123
MAX_LENGTH = 128
BATCH_SIZE = 2
MAX_TOKENS = 1_000_000
MAX_UPDATES = MAX_TOKENS // (BATCH_SIZE * MAX_LENGTH)
VAL_MOD = 10
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.1

torch.manual_seed(SEED)
if device.type == 'cuda':
    torch.cuda.manual_seed_all(SEED)

In [4]:
train_loader, val_loader = create_dataloader_smollm(
    batch_size=BATCH_SIZE, max_length=MAX_LENGTH, val_mod=VAL_MOD,
    seed=SEED, max_tokens=MAX_TOKENS, num_workers=0,
    config_name='cosmopedia-v2', show_progress=True,
    use_rows_api=True, rows_page_size=100,
)

xb, yb = next(iter(train_loader))
# Para este smoke test evitamos recorrer de nuevo SmolLM Corpus para validación.
# La validación real del entrenamiento completo debe usar val_loader.
val_eval_loader = [(xb.clone(), yb.clone())]
assert xb.shape == yb.shape == (BATCH_SIZE, MAX_LENGTH)
assert torch.equal(yb[:, :-1], xb[:, 1:])
print('Micro-batch:', tuple(xb.shape))
print('Tokens máximos del flujo:', f'{MAX_TOKENS:,}')

Dataset: conectando al stream:   0%|          | 0.00/1.00M [00:00<?, ?tok/s]

Micro-batch: (2, 128)
Tokens máximos del flujo: 1,000,000


In [5]:
cfg = {**GPT_CONFIG_50M, 'context_length': MAX_LENGTH}
model = GPTModel(cfg)
model.out_head.weight = model.tok_emb.weight
model = model.to(device)

n_params = sum(parameter.numel() for parameter in model.parameters())
assert n_params <= 50_000_000, f'El modelo tiene {n_params:,} parámetros'
print(f'Parámetros entrenables: {n_params:,} ({n_params / 1e6:.2f}M)')

optimizer = torch.optim.AdamW(
    model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
)

Parámetros entrenables: 47,854,080 (47.85M)


In [6]:
model.train()
train_losses = []
val_losses = []
tokens_seen = 0

for update, (inputs, targets) in enumerate(train_loader, start=1):
    
    inputs = inputs.to(device)
    targets = targets.to(device)
    optimizer.zero_grad(set_to_none=True)
    logits = model(inputs)
    loss = F.cross_entropy(logits.flatten(0, 1), targets.flatten())
    
    if not torch.isfinite(loss):
        raise FloatingPointError(f'Pérdida no finita en la actualización {update}')
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    train_losses.append(loss.item())
    tokens_seen += inputs.numel()
    print(f'Actualización {update:02d}/{MAX_UPDATES} | loss {loss.item():.4f}')
    if update >= MAX_UPDATES:
        break

assert len(train_losses) == MAX_UPDATES, 'El stream terminó antes de completar la prueba'
print(f'Entrenamiento de prueba completado: {tokens_seen:,} tokens')

Dataset: conectando al stream:   0%|          | 0.00/1.00M [00:00<?, ?tok/s]

Actualización 01/3906 | loss 333.2494
Actualización 02/3906 | loss 290.6420
Actualización 03/3906 | loss 200.4483
Actualización 04/3906 | loss 134.9501
Actualización 05/3906 | loss 92.0385
Actualización 06/3906 | loss 79.7453
Actualización 07/3906 | loss 73.1984
Actualización 08/3906 | loss 71.1566
Actualización 09/3906 | loss 65.6586
Actualización 10/3906 | loss 60.0069
Actualización 11/3906 | loss 61.4059
Actualización 12/3906 | loss 63.8411
Actualización 13/3906 | loss 56.6312
Actualización 14/3906 | loss 53.2195
Actualización 15/3906 | loss 53.4915
Actualización 16/3906 | loss 52.8716
Actualización 17/3906 | loss 53.6189
Actualización 18/3906 | loss 52.1241
Actualización 19/3906 | loss 49.1507
Actualización 20/3906 | loss 52.8899
Actualización 21/3906 | loss 50.0827
Actualización 22/3906 | loss 48.7864
Actualización 23/3906 | loss 43.5391
Actualización 24/3906 | loss 46.2498
Actualización 25/3906 | loss 47.2432
Actualización 26/3906 | loss 51.5236
Actualización 27/3906 | loss 50.68

In [7]:
model.eval()
val_losses = []
with torch.inference_mode():
    for inputs, targets in val_eval_loader:
        inputs = inputs.to(device)
        targets = targets.to(device)
        logits = model(inputs)
        val_loss = F.cross_entropy(logits.flatten(0, 1), targets.flatten())
        val_losses.append(val_loss.item())

mean_val_loss = sum(val_losses) / len(val_losses)
print(f'Loss inicial: {train_losses[0]:.4f}')
print(f'Loss final: {train_losses[-1]:.4f}')
print(f'Loss de validación: {mean_val_loss:.4f}')

Loss inicial: 333.2494
Loss final: 8.2946
Loss de validación: 7.5916


In [8]:
tokenizer = tiktoken.get_encoding('gpt2')
prompt = text_to_token_ids('Artificial intelligence', tokenizer).to(device)
generated_ids = generate_text_simple(
    model, prompt, max_new_tokens=20, context_size=MAX_LENGTH
)
print('Muestra:', token_ids_to_text(generated_ids.cpu(), tokenizer))

checkpoint_path = PROJECT_ROOT / 'checkpoints' / 'gpt-50m-smoke-test.pt'
checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'config': cfg,
    'updates': len(train_losses),
    'tokens_seen': tokens_seen,
}, checkpoint_path)
print('Checkpoint guardado en:', checkpoint_path)

Muestra: Artificial intelligence.

4. discrimination a time, a time, we will explore how people who a time
Checkpoint guardado en: /workspace/notebooks/checkpoints/gpt-50m-smoke-test.pt


In [3]:
%pip install matplotlib
%pip install numpy

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [6]:
import matplotlib.pyplot as plt




plt.figure(figsize=(12, 8))
plt.plot(train_losses)

plt.title("Train 50M GPT-1M tokens v1")
plt.xlabel("Steps")
plt.ylabel("Train Loss (log-scale)")
plt.yscale("log")
plt.show()

NameError: name 'train_losses' is not defined

<Figure size 1200x800 with 0 Axes>